In [1]:
import random

DIRECTIONS = {
    "up": (0, -1),
    "down": (0, 1),
    "left": (-1, 0),
    "right": (1, 0),
}

class Thing:

    def __repr__(self):
        return f"<{self.__class__.__name__}>"

class Agent(Thing):

    def __init__(self, program=None):
        self.alive = True
        self.performance = 0
        self.program = program

class Environment:

    def __init__(self):
        self.things = []
        self.agents = []

    def add_thing(self, thing, location=None):
        thing.location = location
        self.things.append(thing)
        if isinstance(thing, Agent):
            self.agents.append(thing)

    def list_things_at(self, location, tclass=Thing):
        return [
            t for t in self.things
            if getattr(t, "location", None) == location and isinstance(t, tclass)
        ]

    def delete_thing(self, thing):
        if thing in self.things:
            self.things.remove(thing)

    def run(self, steps=1000, verbose=True):
        for step in range(steps):
            if self.is_done():
                return step
            if verbose:
                print(f"--- Step {step + 1} ---")
            for agent in self.agents:
                if agent.alive:
                    percept = self.percept(agent)
                    action = agent.program(percept)
                    self.execute_action(agent, action, verbose)
        return steps

    def percept(self, agent):
        raise NotImplementedError

    def execute_action(self, agent, action, verbose=True):
        raise NotImplementedError

    def is_done(self):
        return not any(agent.alive for agent in self.agents)

class Food(Thing):
    pass

class Water(Thing):
    pass

class Wall(Thing):
    pass

class Park(Environment):

    def __init__(self, width, height):
        super().__init__()
        self.width = width
        self.height = height

    def is_inbounds(self, loc):
        x, y = loc
        return 0 <= x < self.width and 0 <= y < self.height

    def is_wall(self, loc):
        return len(self.list_things_at(loc, tclass=Wall)) > 0

    def percept(self, agent):
        here = self.list_things_at(agent.location)
        x, y = agent.location
        walkable = {}
        for d, (dx, dy) in DIRECTIONS.items():
            npos = (x + dx, y + dy)
            walkable[d] = self.is_inbounds(npos) and not self.is_wall(npos)
        return {"here": here, "walkable": walkable}

    def execute_action(self, agent, action, verbose=True):
        x, y = agent.location
        if action in DIRECTIONS:
            dx, dy = DIRECTIONS[action]
            newloc = (x + dx, y + dy)
            if self.is_inbounds(newloc) and not self.is_wall(newloc):
                agent.location = newloc
                agent.visited.add(newloc)
                if verbose:
                    print(f"BlindDog เดินไปทาง '{action}' -> ตำแหน่ง {newloc}")
            elif verbose:
                print(f"BlindDog ชนกำแพง/ขอบสวน ตอนพยายามเดินไปทาง '{action}'")
        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)
            if items:
                self.delete_thing(items[0])
                agent.performance += 10
                if verbose:
                    print(f"BlindDog กินอาหารที่ {agent.location} (performance={agent.performance})")
        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)
            if items:
                self.delete_thing(items[0])
                agent.performance += 5
                if verbose:
                    print(f"BlindDog ดื่มน้ำที่ {agent.location} (performance={agent.performance})")

    def is_done(self):
        return not any(isinstance(t, Food) or isinstance(t, Water) for t in self.things)

class BlindDog(Agent):

    def __init__(self, start_location):
        super().__init__(program=self.decide)
        self.location = start_location
        self.visited = {start_location}
        self.backtrack_count = 0

    def decide(self, percept):
        here = percept["here"]
        walkable = percept["walkable"]

        if any(isinstance(t, Food) for t in here):
            return "eat"
        if any(isinstance(t, Water) for t in here):
            return "drink"

        x, y = self.location
        unvisited_dirs = [
            d for d, (dx, dy) in DIRECTIONS.items()
            if walkable[d] and (x + dx, y + dy) not in self.visited
        ]

        if unvisited_dirs:
            return random.choice(unvisited_dirs)

        walkable_dirs = [d for d, ok in walkable.items() if ok]
        if walkable_dirs:
            self.backtrack_count += 1
            return random.choice(walkable_dirs)

        return "stay"

def print_park(park, dog):
    for y in range(park.height):
        row = []
        for x in range(park.width):
            loc = (x, y)
            if loc == dog.location:
                row.append("D")
            elif park.list_things_at(loc, tclass=Wall):
                row.append("#")
            elif park.list_things_at(loc, tclass=Food):
                row.append("F")
            elif park.list_things_at(loc, tclass=Water):
                row.append("W")
            elif loc in dog.visited:
                row.append("*")
            else:
                row.append(".")
        print(" ".join(row))
    print()

def build_park(width=8, height=8, num_food=3, num_water=3, num_walls=10, seed=None):
    if seed is not None:
        random.seed(seed)

    park = Park(width, height)
    dog = BlindDog(start_location=(0, 0))
    park.add_thing(dog, dog.location)

    occupied = {dog.location}

    def random_free_loc():
        while True:
            loc = (random.randint(0, width - 1), random.randint(0, height - 1))
            if loc not in occupied:
                occupied.add(loc)
                return loc

    for _ in range(num_walls):
        park.add_thing(Wall(), random_free_loc())
    for _ in range(num_food):
        park.add_thing(Food(), random_free_loc())
    for _ in range(num_water):
        park.add_thing(Water(), random_free_loc())

    return park, dog

def main():
    park, dog = build_park(
        width=8, height=8, num_food=3, num_water=3, num_walls=10, seed=None
    )

    print("=== BlindDog เริ่มสำรวจ Park ===")
    print(f"เริ่มที่ {dog.location} | สวนขนาด {park.width}x{park.height}")
    print(f"อาหาร {num_food_in(park)} ชิ้น, น้ำ {num_water_in(park)} แก้ว, กำแพง {num_wall_in(park)} จุด\n")
    print_park(park, dog)

    steps_used = park.run(steps=300, verbose=False)

    print("=== ผลลัพธ์สุดท้าย ===")
    print(f"ใช้ทั้งหมด {steps_used} steps")
    print(f"Performance (คะแนนรวม): {dog.performance}")
    print(f"จำนวนครั้งที่ต้องเดินย้อนกลับ (backtrack): {dog.backtrack_count}")
    print(f"จำนวนตำแหน่งที่สำรวจไปแล้ว: {len(dog.visited)} จาก {park.width * park.height} ช่อง\n")

    print("แผนที่สุดท้าย ('D'=สุนัข, '#'=กำแพง, 'F'=อาหาร, 'W'=น้ำ, '*'=เคยผ่าน):")
    print_park(park, dog)

def num_food_in(park):
    return sum(1 for t in park.things if isinstance(t, Food))

def num_water_in(park):
    return sum(1 for t in park.things if isinstance(t, Water))

def num_wall_in(park):
    return sum(1 for t in park.things if isinstance(t, Wall))

if __name__ == "__main__":
    main()

=== BlindDog เริ่มสำรวจ Park ===
เริ่มที่ (0, 0) | สวนขนาด 8x8
อาหาร 3 ชิ้น, น้ำ 3 แก้ว, กำแพง 10 จุด

D . . # . . . .
. . . . . . . .
# . . # . . . #
. . W . . F . .
. . . . W # F .
. . W . . . . #
. . # . # . . #
F . . # . . . .

=== ผลลัพธ์สุดท้าย ===
ใช้ทั้งหมด 78 steps
Performance (คะแนนรวม): 45
จำนวนครั้งที่ต้องเดินย้อนกลับ (backtrack): 33
จำนวนตำแหน่งที่สำรวจไปแล้ว: 40 จาก 64 ช่อง

แผนที่สุดท้าย ('D'=สุนัข, '#'=กำแพง, 'F'=อาหาร, 'W'=น้ำ, '*'=เคยผ่าน):
* . . # . . . .
* * . . . . . .
# * * # * * * #
* * * * * D * *
* * * * * # * *
* * * * * * * #
* * # * # * * #
* * . # * * * .

